# Free rigid-body rotation (tennis-racket effect)

*mechanics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/rigid_body_rotation)

Euler's equations for a torque-free box, written with the inertia tensor $I_{ij}$ and the Levi-Civita tensor $\varepsilon_{ijk}$. Spin about the longest or shortest axis is stable; spin about the intermediate axis flips over and over (Dzhanibekov effect).

**Vector form**

$$\mathsf I\,\dot{\boldsymbol\omega} + \boldsymbol\omega\times(\mathsf I\,\boldsymbol\omega) = 0,\qquad \dot{\mathsf R} = \mathsf R\,[\boldsymbol\omega]_\times$$

**Index (tensor) form**

$$I_{ij}\,\dot\omega_j + \varepsilon_{ijk}\,\omega_j\,I_{kl}\,\omega_l = 0,\qquad \dot R_{ij} = R_{ik}\,\varepsilon_{klj}\,\omega_l$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $I_{ij}$ | `box_inertia(M, a, b, c)` | (3, 3) | inertia tensor of a solid box, $(M/12)\operatorname{diag}(b^2+c^2,\ a^2+c^2,\ a^2+b^2)$ |
| $\omega_i$ | `y[3]` | (3,) | angular velocity in body-frame components |
| $\varepsilon_{ijk}$ | `levi_civita()` | (3, 3, 3) | Levi-Civita tensor: the cross product as a tensor |
| $\varepsilon_{ijk}\omega_j I_{kl}\omega_l$ | `einsum('ijk,j,kl,l->i', eps, w, I, w)` | (3,) | $j, k, l$ summed: $\boldsymbol\omega\times(I\boldsymbol\omega)$ |
| $\dot\omega_j$ | `torch.linalg.solve(I, torque)` | (3,) | solve $I_{ij}\dot\omega_j = -\varepsilon_{ijk}\omega_j I_{kl}\omega_l$ |
| $R_{ij}$ | `y[:3]` | (3, 3) | orientation: body → world rotation matrix |
| $\tfrac12\omega_i I_{ij}\omega_j$ | `einsum('...i,ij,...j->...', w, I, w)` | () | rotational kinetic energy, conserved |

With no torque both the energy $\tfrac12\boldsymbol\omega\cdot I\boldsymbol\omega$ and the size of $\mathbf L = I\boldsymbol\omega$ are conserved, so $\boldsymbol\omega$ must move on the intersection of an ellipsoid of constant energy and a sphere of constant $\lvert\mathbf L\rvert$ (in $\mathbf L$-space). Near the major and minor axes these curves are small loops — stable; near the intermediate axis they pass through a saddle, so a tiny wobble grows and the box turns over, then turns back. The world-frame angular momentum $RI\boldsymbol\omega$ (drawn as the fixed arrow) never moves. Parameters $a, b, c$ are sorted so that $a \ge b \ge c$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/rigid_body_rotation/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    a, b, c = sorted([params["a"], params["b"], params["c"]], reverse=True)  # a ≥ b ≥ c
    inertia = tensors.box_inertia(1.0, a, b, c, dev)
    y0 = tensors.initial_state(params["spin_axis"], params["spin"], params["perturbation"], dev)

    sol = solve.solve(y0, inertia, params["t_max"])
    T = equation.kinetic_energy(sol["omega"], inertia)  # (T,)
    L_sq = equation.angular_momentum(sol["omega"], inertia).pow(2).sum(-1)  # (T,)

    print("principal moments I_11, I_22, I_33 =", [round(x, 4) for x in inertia.diagonal().tolist()])
    drift_T = ((T - T[0]) / T[0]).abs().max()
    drift_L = ((L_sq - L_sq[0]) / L_sq[0]).abs().max()
    print(f"relative drift: energy {drift_T:.1e}, |L|² {drift_L:.1e}")
    w = sol["omega"][:, tensors.AXES[params["spin_axis"]]]
    print(f"spin component: min {w.min():.2f}, max {w.max():.2f} rad/s (a sign change means the body flipped)")
    return plot.animate(sol, inertia, (a, b, c), out_dir / "rigid_body_rotation.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/rigid_body_rotation/equation.py
r"""Physics → tensor translation.

A rigid body spinning freely (no torque) keeps its angular momentum \mathbf L = \mathsf I \boldsymbol\omega fixed
in space. Seen from the body's own axes, \mathbf L turns, and Euler's equations follow:

    \mathsf I \dot{\boldsymbol\omega} + \boldsymbol\omega \times (\mathsf I \boldsymbol\omega) = 0 .

In index notation (body-frame components i, j, k, l ∈ {1, 2, 3}, repeated indices summed):

    I_{ij} \dot\omega_j + \varepsilon_{ijk} \, \omega_j \, I_{kl} \omega_l = 0

The cross product is the Levi-Civita tensor ε_ijk contracted with two vectors — one einsum, 'ijk,j,kl,l->i'.
The orientation R_ij (body → world) changes as

    \dot R_{ij} = R_{ik} \, \varepsilon_{klj} \, \omega_l          (\dot R = R [\omega]_\times)

Conserved: the kinetic energy  T = ½ ω_i I_ij ω_j  and  |L|² = (I_ij ω_j)(I_ik ω_k),
and in the world frame the whole vector  L^{world}_i = R_ij I_jk ω_k.
"""

import torch


def levi_civita(dtype: torch.dtype = torch.float64, device: torch.device | None = None) -> torch.Tensor:
    """ε_ijk of shape (3, 3, 3): +1 for even permutations of (0, 1, 2), −1 for odd ones, 0 otherwise."""
    eps = torch.zeros(3, 3, 3, dtype=dtype, device=device)
    for i, j, k in [(0, 1, 2), (1, 2, 0), (2, 0, 1)]:
        eps[i, j, k], eps[i, k, j] = 1.0, -1.0
    return eps


def gyroscopic_torque(omega: torch.Tensor, inertia: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""-ε_{ijk} ω_j I_{kl} ω_l  (j, k, l summed): the 'torque' that turns ω in the body frame.   → (3,)"""
    return -torch.einsum("ijk,j,kl,l->i", eps, omega, inertia, omega)


def rhs(y: torch.Tensor, inertia: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    """dy/dt for the state y of shape (4, 3): y[:3] = orientation R (3, 3), y[3] = body-frame ω (3,)."""
    R, omega = y[:3], y[3]
    omega_dot = torch.linalg.solve(inertia, gyroscopic_torque(omega, inertia, eps))  # I_ij ω̇_j = torque_i
    R_dot = torch.einsum("ik,klj,l->ij", R, eps, omega)  # Ṙ = R [ω]×
    return torch.cat([R_dot, omega_dot[None]], dim=0)


def kinetic_energy(omega: torch.Tensor, inertia: torch.Tensor) -> torch.Tensor:
    r"""T = ½ ω_i I_{ij} ω_j.   omega: (..., 3) → (...)"""
    return 0.5 * torch.einsum("...i,ij,...j->...", omega, inertia, omega)


def angular_momentum(omega: torch.Tensor, inertia: torch.Tensor) -> torch.Tensor:
    r"""Body-frame L_i = I_{ij} ω_j.   (..., 3) → (..., 3)"""
    return torch.einsum("ij,...j->...i", inertia, omega)


def world_angular_momentum(R: torch.Tensor, omega: torch.Tensor, inertia: torch.Tensor) -> torch.Tensor:
    r"""L^{world}_i = R_{ij} I_{jk} ω_k — a fixed vector when there is no torque.   (..., 3) → (..., 3)"""
    return torch.einsum("...ij,...j->...i", R, angular_momentum(omega, inertia))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/rigid_body_rotation/tensors.py
"""Build the concrete tensors: the box's inertia tensor and the initial state (orientation, spin).

float64 keeps the conserved quantities accurate to many digits over thousands of steps.
"""

import torch

DTYPE = torch.float64
AXES = {"minor": 0, "intermediate": 1, "major": 2}  # body axis index for each choice


def box_inertia(mass: float, a: float, b: float, c: float, device: torch.device | None = None) -> torch.Tensor:
    """I_ij of a solid box with sides a > b > c along body axes 1, 2, 3: shape (3, 3).

    I = (M / 12) diag(b² + c², a² + c², a² + b²)

    The longest side a gives the smallest moment (minor axis 1), the shortest side c the largest (major axis 3).
    """
    moments = mass / 12 * torch.tensor([b**2 + c**2, a**2 + c**2, a**2 + b**2], dtype=DTYPE, device=device)
    return torch.diag(moments)


def initial_state(axis: str, spin: float, perturbation: float, device: torch.device | None = None) -> torch.Tensor:
    """y0 of shape (4, 3): orientation R = identity (rows 0–2) and body-frame ω (row 3).

    ω is mostly along the chosen axis, plus a small nudge of size perturbation · spin on the other two.
    """
    omega = perturbation * spin * torch.ones(3, dtype=DTYPE, device=device)  # (3,)
    omega[AXES[axis]] = spin
    R = torch.eye(3, dtype=DTYPE, device=device)  # (3, 3)
    return torch.cat([R, omega[None]], dim=0)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/rigid_body_rotation/solve.py
"""Integrate Euler's equations together with the orientation, using 4th-order Runge–Kutta."""

import torch

from problems._common.integrators import integrate

from .equation import levi_civita, rhs


def solve(y0: torch.Tensor, inertia: torch.Tensor, t_max: float, dt: float = 0.004) -> dict[str, torch.Tensor]:
    """Return times (T,), orientations R (T, 3, 3) and body-frame angular velocities ω (T, 3)."""
    eps = levi_civita(y0.dtype, y0.device)  # (3, 3, 3)
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, inertia, eps), y0, dt, n_steps)  # (T, 4, 3)
    times = torch.arange(n_steps + 1, device=y0.device, dtype=y0.dtype) * dt
    return {"times": times, "R": traj[:, :3], "omega": traj[:, 3]}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/rigid_body_rotation/plot.py
"""3D animation of the spinning box (world frame) next to its body-frame angular velocity ω_i(t)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import world_angular_momentum

# the 6 faces of a box, as indices into the 8 corners built below
FACES = [[0, 1, 3, 2], [4, 5, 7, 6], [0, 1, 5, 4], [2, 3, 7, 6], [0, 2, 6, 4], [1, 3, 7, 5]]
FACE_COLORS = [ACCENT[0], ACCENT[0], ACCENT[1], ACCENT[1], ACCENT[2], ACCENT[2]]  # opposite faces match


def animate(sol: dict, inertia: torch.Tensor, sides: tuple[float, float, float], path: Path) -> Path:
    t = sol["times"].cpu().numpy()  # (T,)
    R = sol["R"].cpu().numpy()  # (T, 3, 3) body → world
    omega = sol["omega"].cpu().numpy()  # (T, 3) body frame
    L_world = world_angular_momentum(sol["R"][0], sol["omega"][0], inertia).cpu().numpy()  # (3,) fixed in space

    half = np.array(sides) / 2
    signs = np.array([[x, y, z] for x in (-1, 1) for y in (-1, 1) for z in (-1, 1)])  # (8, 3)
    corners = signs * half  # (8, 3) in body coordinates
    reach = np.linalg.norm(half) * 1.05

    fig = plt.figure(figsize=(7, 4))
    ax_box = fig.add_subplot(1, 2, 1, projection="3d")
    ax_w = fig.add_subplot(1, 2, 2)

    # --- left: the box in the world frame, with the fixed angular momentum ---
    for set_lim in (ax_box.set_xlim, ax_box.set_ylim, ax_box.set_zlim):
        set_lim(-reach, reach)
    ax_box.set_box_aspect((1, 1, 1))
    ax_box.set_xticklabels([])
    ax_box.set_yticklabels([])
    ax_box.set_zticklabels([])
    box = Poly3DCollection([corners[f] for f in FACES], facecolors=FACE_COLORS, edgecolor="#111827", alpha=0.85)
    ax_box.add_collection3d(box)
    L_dir = 1.3 * reach * L_world / np.linalg.norm(L_world)
    ax_box.plot([0, L_dir[0]], [0, L_dir[1]], [0, L_dir[2]], color="#111827", lw=2)
    ax_box.text(*L_dir, "L", fontsize=10)
    clock = ax_box.text2D(0.02, 0.95, "", transform=ax_box.transAxes, fontsize=9)

    # --- right: body-frame ω components ---
    labels = ["ω₁ minor", "ω₂ intermediate", "ω₃ major"]
    ax_w.set_xlim(t[0], t[-1])
    ax_w.set_ylim(omega.min() - 0.5, omega.max() + 0.5)
    ax_w.set_xlabel("t (s)")
    ax_w.set_ylabel("ω in body frame (rad/s)")
    lines = [ax_w.plot([], [], lw=1.4, color=ACCENT[i], label=labels[i])[0] for i in range(3)]
    ax_w.legend(fontsize=7, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False)

    def update(step: int):
        world = corners @ R[step].T  # (8, 3) x^{world}_i = R_ij x^{body}_j for every corner
        box.set_verts([world[f] for f in FACES])
        for i, line in enumerate(lines):
            line.set_data(t[: step + 1], omega[: step + 1, i])
        clock.set_text(f"t = {t[step]:.2f} s")

    ax_box.view_init(elev=20, azim=-50)
    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 120), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'spin_axis': 'intermediate',
    'spin': 5.0,
    'perturbation': 0.01,
    'a': 3.0,
    'b': 2.0,
    'c': 1.0,
    't_max': 12.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.rigid_body_rotation")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.rigid_body_rotation.main")
problem_dir = Path("problems/mechanics/rigid_body_rotation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
